# Convert Unstructured text into a Structured Object

In [1]:
from dotenv import load_dotenv
import os

## Setup API Keys

In [2]:
load_dotenv()

SARVAM_API_KEY = os.getenv('SARVAM_API_KEY')
assert SARVAM_API_KEY is not None, 'Could NOT load SARVAM_API_KEY from .env'     

## Use Case - Extract Data from an Incident Ticket

### Create a fictitious ticket summary

In [37]:
ticket_summary = """
INC7126761 - Financial Advisor - John Doe (id - jdoe123) from ABC Financial Services reporting slowness 
in the Options Trading Screen.
The screen did NOT load despite repeated attempts around 11:50 AM on 22nd Sep 2026.
Please investigate
""".strip()

### Define Output Structure

In [45]:
from pydantic import BaseModel

class IncidentTicket(BaseModel):
    ticket_id: str
    date_YYYYMMDD: str
    time: str
    company_name: str
    customer_name: str
    user_id: str
    account_numbers: list[str]
    other_identifiers: list[str]
    application_function: str

### Call Sarvam AI using Langchain plugin

In [46]:
from langchain_sarvam import ChatSarvam

In [47]:
llm = ChatSarvam(model="sarvam-105b-conversations")

In [48]:
llm.invoke('How are you ?').content

'I’m doing well, thanks for asking. How about you?'

### Extract Data using Lanchain's Structured Output

In [49]:
structured_llm = llm.with_structured_output(IncidentTicket)

In [53]:
result = structured_llm.invoke(f'Extract the fields from the ticket:\n{ticket_summary}')

In [54]:
print(result.model_dump_json(indent=2))

{
  "ticket_id": "INC7126761",
  "date_YYYYMMDD": "20260922",
  "time": "11:50 AM",
  "company_name": "ABC Financial Services",
  "customer_name": "John Doe",
  "user_id": "jdoe123",
  "account_numbers": [],
  "other_identifiers": [
    "Financial Advisor"
  ],
  "application_function": "Options Trading Screen"
}


## Use Case 2 - Solve a Math problem

### Define Math problem

In [55]:
math_problem = """List the steps to solve the system of linear equations: 
                    x + y = 5
                    x - y = 1
                """.strip()

### Define Structure for answer

In [56]:
class Step(BaseModel):
    step_num: int
    step: str

class Steps(BaseModel):
    steps: list[Step]

In [57]:
structured_llm = llm.with_structured_output(Steps)

In [58]:
result = structured_llm.invoke(
    math_problem
)

In [59]:
print(result.model_dump_json(indent=2))

{
  "steps": [
    {
      "step_num": 1,
      "step": "Write down the given system of equations: x + y = 5 and x - y = 1."
    },
    {
      "step_num": 2,
      "step": "Choose a method to solve the system. Here, the elimination method is convenient because the y terms have opposite signs."
    },
    {
      "step_num": 3,
      "step": "Add the two equations together to eliminate y: (x + y) + (x - y) = 5 + 1, which simplifies to 2x = 6."
    },
    {
      "step_num": 4,
      "step": "Solve for x by dividing both sides by 2: x = 3."
    },
    {
      "step_num": 5,
      "step": "Substitute the value x = 3 into one of the original equations (e.g., x + y = 5) to find y."
    },
    {
      "step_num": 6,
      "step": "Solve for y: 3 + y = 5, so y = 5 − 3 = 2."
    },
    {
      "step_num": 7,
      "step": "State the solution: x = 3 and y = 2, or as an ordered pair (3, 2)."
    },
    {
      "step_num": 8,
      "step": "Verify the solution by substituting x = 3 and y = 2 int